# Building Footprint Extraction 

This section will show how to extract building footprints from a raster dataset.

## Import the geoai Package

This notebook demonstrates building footprint extraction from aerial/satellite imagery using the geoai library. The geoai package provides pre-trained deep learning models specifically designed for extracting building footprints from raster imagery (such as NAIP, Sentinel-2, or high-resolution aerial photos).

The workflow consists of:
1. **Data Preparation** - Download sample NAIP imagery and ground truth vector data
2. **Model Initialization** - Load a pre-trained building footprint extraction model
3. **Building Extraction** - Two approaches:
   - **Option 1 (Two-step)**: Generate binary masks → Convert masks to vector polygons
   - **Option 2 (End-to-end)**: Direct raster-to-vector extraction
4. **Post-processing** - Regularize extracted polygons to improve geometric quality
5. **Visualization** - Interactive maps and static visualizations

In [1]:
# Import the geoai package - provides tools for geospatial AI tasks including building footprint extraction
import geoai

In [2]:
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train.tif"
)
vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"

raster_path = geoai.download_file(raster_url)
vector_path = geoai.download_file(vector_url)

In [3]:
# Move the downloaded files to the target folder

import os
import shutil
from pathlib import Path

# ==== INPUTS ====

target_folder = "data/geoai/building_extraction"  # Enter your target folder path here
datasets = ["naip_train.tif", "naip_train_buildings.geojson"]  # Add dataset names as a list
source_dir = Path(".")  # Current location of datasets

# ==== SCRIPT ====

target_path = Path(target_folder)
target_path.mkdir(parents=True, exist_ok=True)
for dataset_name in datasets:
    source_path = source_dir / dataset_name
    
    if not source_path.exists():
        print(f"Dataset '{dataset_name}' not found in {source_dir}")
        continue
    
    if source_path.is_file():
        files_to_move = [source_path]
    else:
        files_to_move = list(source_path.parent.glob(f"{dataset_name}.*"))
    
    for file_path in files_to_move:
        dest = target_path / file_path.name
        shutil.move(str(file_path), str(dest))
        print(f"Moved: {file_path.name} -> {target_folder}")
        
print("Done!")

Moved: naip_train.tif -> data/geoai/building_extraction
Moved: naip_train_buildings.geojson -> data/geoai/building_extraction
Done!


In [4]:
# Initialize building footprint extraction pretrained model

extractor = geoai.BuildingFootprintExtractor()

Model path not specified, downloading from Hugging Face...


building_footprints_usa.pth:   0%|          | 0.00/176M [00:00<?, ?B/s]

Model downloaded to: /home/bnhr/.cache/huggingface/hub/models--giswqs--geoai/snapshots/089548329c81f128fa12576663e7abdedb5cfa0e/building_footprints_usa.pth


RuntimeError: Failed to load model: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 11.61 GiB of which 78.00 MiB is free. Process 65909 has 8.90 GiB memory in use. Process 67120 has 1.28 GiB memory in use. Including non-PyTorch memory, this process has 454.00 MiB memory in use. Of the allocated memory 196.66 MiB is allocated by PyTorch, and 7.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Extract building footprints
# Option 1: Extract building footprints as raster¶

raster = "data/geoai/building_extraction/naip_train.tif"
# vector = "data/geoai/building_extraction/naip_train_buildings.geojson"

mask_path = extractor.save_masks_as_geotiff(
    raster_path=raster,
    output_path="data/geoai/building_extraction/building_masks.tif",
    confidence_threshold=0.5,
    mask_threshold=0.5,
)


In [ ]:
# Convert raster to vector

gdf = extractor.masks_to_vector(
    mask_path=mask_path,
    output_path="data/geoai/building_extraction/building_masks.geojson",
    simplify_tolerance=1.0,
)

In [ ]:
# Option 2: Extract building footprints as vector

output_path = "data/geoai/building_extraction/naip_buildings.geojson"
gdf = extractor.process_raster(
    raster,
    output_path="data/geoai/building_extraction/naip_buildings.geojson",
    batch_size=4,
    confidence_threshold=0.5,
    overlap=0.25,
    nms_iou_threshold=0.5,
    min_object_area=100,
    max_object_area=None,
    mask_threshold=0.5,
    simplify_tolerance=1.0,
)

In [ ]:
# Regularize building footprints
gdf_regularized = extractor.regularize_buildings(
    gdf=gdf,
    min_area=100,
    angle_threshold=15,
    orthogonality_threshold=0.3,
    rectangularity_threshold=0.7,
)

In [ ]:
# Visualize building footprints

gdf.head()

In [ ]:
geoai.view_vector_interactive(
    gdf, column="confidence", layer_name="Building", tiles="Satellite"
)


In [ ]:
geoai.view_vector_interactive(
    gdf, column="confidence", layer_name="Building", tiles=raster_url
)

In [ ]:
geoai.view_vector_interactive(
    gdf_regularized, column="confidence", layer_name="Building", tiles=raster_url
)

In [ ]:
extractor.visualize_results(raster, gdf, output_path="data/geoai/building_extraction/naip_buildings.png")

In [ ]:
extractor.visualize_results(
    raster, gdf_regularized, output_path="data/geoai/building_extraction/naip_buildings_regularized.png"
)